# 10 — final-token causal endpoints (the one methodologically-motivated GPU run)

`analysis_plan.md` §4 fixes the direction on the **final prompt token**; the
committed causal pipeline ablates the **mean of the last 5 tokens** (audit
RED-1). `cos(final, pooled) ≈ 0.76–0.87` at the ablation layers — close but
not the same direction. Everything descriptive (z_C, factorial, geometry,
source-robustness) is already final-token; CF1 is behavioral and
pooling-independent. **This run regenerates only the ablation-based
endpoints on final-token:** CF2 held-out, 5-fold cross-fit, the 2×2, the
circularity check, and the quadrant-C McNemar.

Set `SCOPE` in cell 1:
- `"m3_anchor"` — M3 held-out CF2 only (~1 h). Does the preregistered anchor
  replicate under the preregistered pooling? If yes, RED-1 becomes "robust
  to the pooling choice."
- `"full"` — 4 branches, held-out + cross-fit, both judges (~4–5 h;
  generation §4 and judging §6 are separate, resumable, split across sessions).

Needs: T4 runtime, Colab secret `HF_TOKEN` (with `google/gemma-2b` **and**
`allenai/wildguard` accepted), your `dpo_v2` Drive folder.

## 1. Config + clone + Drive + HF

In [ ]:
SCOPE = "m3_anchor"      # "m3_anchor" | "full"
assert SCOPE in ("m3_anchor", "full")

import os, sys, subprocess, glob, json
from pathlib import Path
from collections import Counter

REPO = "https://github.com/urosavurdic/dpo-safety-representations.git"
BR   = "agent/c-quadrant-end-to-end-e0e2317a"
if not os.path.isdir("dpo-safety-representations"):
    subprocess.run(["git", "clone", REPO], check=True)
os.chdir("dpo-safety-representations")
subprocess.run(["git", "fetch", "origin", "--quiet"], check=True)
subprocess.run(["git", "checkout", "-B", BR, "origin/" + BR], check=True)   # branch tip
print("HEAD:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

from google.colab import drive, userdata
drive.mount("/content/drive")
cands = (["/content/drive/MyDrive/dpo_v2"]
         + sorted(glob.glob("/content/drive/.shortcut-targets-by-id/*/dpo_v2"))
         + sorted(glob.glob("/content/drive/Shareddrives/*/dpo_v2")))
REAL = next((c for c in cands
             if glob.glob(os.path.join(c, "results", "activations", "*_final.npy"))), None)
assert REAL, "no dpo_v2 folder with activations found:\n  " + "\n  ".join(cands)
os.environ["DPO_DRIVE_ROOT"] = REAL
from src.colab_persist import bind, status_line
print(status_line(bind(persist_hf_cache=False)))

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])
print("HF ok")

BRANCHES = ["M3"] if SCOPE == "m3_anchor" else ["M3", "M3_direct", "M3_alt", "M3_direct_alt"]
RUNS = [("held-out", "", None)] + ([("xfit5", "_xfit5", 360)] if SCOPE == "full" else [])
print("SCOPE:", SCOPE, "| branches:", BRANCHES, "| runs:", [r[0] for r in RUNS])

## 2. Env fix + fail-fast (before any GPU time)

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "bitsandbytes", "accelerate"], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
import importlib.util
if importlib.util.find_spec("torchao") is not None:
    import peft.import_utils as _piu
    _piu.is_torchao_available = lambda *a, **k: False
    try:
        import peft.tuners.lora.torchao as _plt
        _plt.is_torchao_available = lambda *a, **k: False
    except Exception:
        pass
import torch, transformers, peft
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "| transformers", transformers.__version__, "| peft", peft.__version__)
assert torch.cuda.is_available(), "no GPU — set the runtime to a T4."

from src.training.model import load_stage_model
_m, _t = load_stage_model("M3")
print("load_stage_model('M3') OK —", type(_m).__name__)
del _m, _t
import gc; gc.collect(); torch.cuda.empty_cache()

## 3. STATUS — what exists, what's left (no GPU)

In [ ]:
import numpy as np
BENCH = sorted(glob.glob("data/frozen_v2/benchmark_v2_*.jsonl"))[-1]

print("=== 654-row _final activations (needed to build the final-token direction) ===")
ok = True
for b in BRANCHES:
    mp, fp = f"results/activations/{b}_metadata.json", f"results/activations/{b}_final.npy"
    if not (os.path.exists(mp) and os.path.exists(fp)):
        print(f"  {b:16s} MISSING"); ok = False; continue
    m = json.load(open(mp, encoding="utf-8", errors="replace"))
    n = np.load(fp, mmap_mode="r").shape[0]
    sp = sum(1 for r in m if r.get("split"))
    good = len(m) == 654 and n == 654 and sp == 300
    print(f"  {b:16s} meta={len(m)} npy={n} splits={sp}  {'OK' if good else '<-- STALE'}")
    ok = ok and good
assert ok, "activations not all 654-row — cannot build the final-token direction."

def nrows(p):
    try: return len(json.load(open(p, encoding="utf-8", errors="replace")))
    except Exception: return None

print("\n=== final-token causal generations on Drive ===")
todo = []
for b in BRANCHES:
    for label, sfx, want in RUNS:
        p = f"results/raw/causal_ablation_v2_{b}_L24-28{sfx}_finaltoken.json"
        n = nrows(p)
        done = (n is not None) and (n == want if want else n > 300)
        print(f"  {b:16s} {label:9s} {str(n):>6} rows  {'OK' if done else 'TODO'}")
        if not done: todo.append((b, label, sfx, want))

print("\n=== final-token judged files ===")
jf = sorted(glob.glob("results/final_token_repair/judges/behavioral_judges_v2_*.json"))
if jf:
    d = json.load(open(jf[-1]))
    print(f"  newest: {os.path.basename(jf[-1])}  judge_status={d.get('judge_status')}")
else:
    print("  none yet")

print("\n" + "=" * 56 + "\nREMAINING WORK\n" + "=" * 56)
for b, label, *_ in todo: print(f"  GENERATE  {b:16s} {label}")
if not todo: print("  generation: COMPLETE for this SCOPE")
print("  JUDGE §6 — both models; POST §8 — always re-run")
TODO = todo

## 4. GENERATE — final-token held-out (+ cross-fit if SCOPE=full)

Resumable: `v2_pipeline` skips a branch whose bound output exists. `git push`
after each branch; Drive already has everything (`results/` is symlinked).

In [ ]:
def ckpt(label):
    try:
        subprocess.run(["git", "config", "user.email", "noreply@anthropic.com"], check=True)
        subprocess.run(["git", "config", "user.name", "finaltoken10 (Colab)"], check=True)
        add = [f for f in glob.glob("results/raw/causal_ablation_v2_*finaltoken*") if os.path.isfile(f)]
        subprocess.run(["git", "add", "--"] + add, check=True, capture_output=True)
        if subprocess.check_output(["git", "diff", "--cached", "--name-only"], text=True).strip():
            subprocess.run(["git", "commit", "-m", f"final-token causal (Colab): {label}"],
                           check=True, capture_output=True)
            subprocess.run(["git", "pull", "--rebase", "origin", BR], check=False, capture_output=True)
            r = subprocess.run(["git", "push", "origin", BR], capture_output=True, text=True)
            print(f"    [ckpt {label}] push rc={r.returncode}"
                  + ("" if r.returncode == 0 else "  (Drive holds it)"))
    except Exception as ex:
        print(f"    [ckpt {label}] git skipped ({ex}) — Drive holds it")

BASE = [sys.executable, "-m", "src.analysis.v2_pipeline", "causal", "--pooling", "final_token"]
for b in BRANCHES:
    for label, sfx, want in RUNS:
        cmd = BASE + ["--stage", b] + (["--cross-fit", "5"] if label == "xfit5" else [])
        print(f"\n>>> {b} {label}  (final-token)"); sys.stdout.flush()
        r = subprocess.run(cmd)
        assert r.returncode == 0, f"generation FAILED: {b} {label}"
        p = f"results/raw/causal_ablation_v2_{b}_L24-28{sfx}_finaltoken.json"
        rows = json.load(open(p, encoding="utf-8", errors="replace"))
        conds = dict(Counter(x.get("stage") for x in rows))
        print(f"    {p}  {len(rows)} rows  {conds}")
        if want:
            assert len(rows) == want, f"{b} {label}: expected {want}, got {len(rows)}"
        else:
            assert len(rows) > 300 and len(conds) == 3, f"{b} held-out: shape {conds}"
    ckpt(b)
print("\nGENERATION COMPLETE for SCOPE =", SCOPE)

## 5. Judge probe — StrongREJECT + WildGuard, fp16 then 4-bit fallback

In [ ]:
import gc
from src.analysis.behavioral_judges import (LazyModelJudge, DEFAULT_STRONGREJECT_MODEL,
                                             DEFAULT_WILDGUARD_MODEL)

def probe(name, mid, mode, four_bit):
    j = LazyModelJudge(name, mid, load_4bit=four_bit, allow_download=True, mode=mode)
    ok = j.try_load(); err = None if ok else j.load_error
    try: j.unload()
    except Exception: pass
    del j; gc.collect(); torch.cuda.empty_cache()
    return ok, err

LOAD_4BIT = False
for nm, mid, m_ in (("strong_reject", DEFAULT_STRONGREJECT_MODEL, "score_1_to_5"),
                    ("wildguard",     DEFAULT_WILDGUARD_MODEL,    "generate")):
    ok, err = probe(nm, mid, m_, four_bit=False)
    print(f"{nm:14s} fp16: {'OK' if ok else 'FAIL - ' + str(err)[:150]}")
    if not ok:
        ok4, err4 = probe(nm, mid, m_, four_bit=True)
        print(f"{nm:14s} 4bit: {'OK' if ok4 else 'FAIL - ' + str(err4)[:150]}")
        assert ok4, f"{nm} loads in neither fp16 nor 4-bit — check the HF licence for {mid}."
        LOAD_4BIT = True
print("\nLOAD_4BIT =", LOAD_4BIT, " (both judges will run)")

## 6. JUDGE — both models, one at a time, resume from any prior file

In [ ]:
Path("results/final_token_repair/manifests").mkdir(parents=True, exist_ok=True)
Path("results/final_token_repair/judges").mkdir(parents=True, exist_ok=True)

ft = sorted(f for f in glob.glob("results/raw/causal_ablation_v2_*L24-28*finaltoken*.json")
            if not f.endswith("_binding.json"))
assert ft, "no final-token causal files — run §4 first"
b0 = json.load(open(ft[0].replace(".json", "_binding.json")))
mp = "results/final_token_repair/manifests/consolidated_finaltoken.json"
json.dump({"kind": "consolidated_response_manifest", "pooling": "final_token",
           "benchmark_sha256": b0.get("benchmark_sha256"),
           "split_manifest_sha256": b0.get("split_manifest_sha256"),
           "entries": [{"response_file": f, "binding_file": f.replace(".json", "_binding.json")}
                       for f in ft]}, open(mp, "w"), indent=2)

prior = sorted(glob.glob("results/final_token_repair/judges/behavioral_judges_v2_*.json"))
resume = prior[-1] if prior else None
print("resume-from:", os.path.basename(resume) if resume else "(none)")

cmd = [sys.executable, "-m", "src.analysis.behavioral_judges",
       "--response-manifest", mp, "--run-live", "--allow-download",
       "--require-binding", "--reject-legacy",
       "--out-dir", "results/final_token_repair/judges"]
if not LOAD_4BIT: cmd += ["--no-4bit"]
if resume:        cmd += ["--resume-from", resume]
before = set(prior)
r = subprocess.run(cmd)
assert r.returncode == 0, "judge failed — read the traceback"
jf = sorted(set(glob.glob("results/final_token_repair/judges/behavioral_judges_v2_*.json")) - before)
jf = jf[-1] if jf else prior[-1]
print("\njudged file:", jf, round(os.path.getsize(jf) / 1e6, 1), "MB")
ckpt("judged")

## 7. Verify — every in-scope row scored by BOTH judges

In [ ]:
data = json.load(open(jf)); recs = data.get("records", []); js = data.get("judge_status", {})
st = lambda r: r.get("stage") or r.get("condition") or ""
sr = lambda r: (r.get("strong_reject") or {}).get("judge_status") == "scored"
wg = lambda r: (r.get("wildguard") or {}).get("judge_status") == "scored"
adA = [r for r in recs if "_ft_" in st(r) and "xfit" not in st(r) and r.get("quadrant") in (None, "A")]
xf  = [r for r in recs if "ft_xfit" in st(r)]
print(f"ft_ held-out quA  {len(adA):5d}  SR={sum(map(sr,adA)):5d}  WG={sum(map(wg,adA)):5d}")
print(f"ft_xfit          {len(xf):5d}  SR={sum(map(sr,xf)):5d}  WG={sum(map(wg,xf)):5d}")
print("judge_status:", js)
assert js.get("strong_reject") == "scored", "StrongREJECT incomplete"
assert js.get("wildguard") == "scored", ("WildGuard incomplete — it is the preregistered "
    "independent cross-check; re-run §6.")
assert sum(map(sr, adA)) >= 25 and sum(map(wg, adA)) >= 25, "held-out A not scored by both"
if SCOPE == "full":
    assert sum(map(sr, xf)) >= 1400 and sum(map(wg, xf)) >= 1400, "cross-fit not fully scored"
print("\nOK — both judges complete.")

## 8. POST — final-token endpoints + quad-C McNemar + pooled-vs-final table

In [ ]:
SUM = "results/final_token_repair/summaries"; Path(SUM).mkdir(parents=True, exist_ok=True)
EP = SUM + "/final_token_endpoints.json"
r = subprocess.run([sys.executable, "-m", "src.analysis.confirmatory_behavioral_endpoints",
                    "--judged", jf, "--benchmark", BENCH,
                    "--condition-infix", "ft_", "--out", EP])
assert r.returncode == 0 and os.path.exists(EP), "POST failed"
e = json.load(open(EP)); pooled = json.load(open("results/summaries/confirmatory_endpoints.json"))

def fmt(x):
    if not x or x.get("cf2") is None: return "n=0"
    return f"{x['cf2']:+.4f} [{x['ci_low']:+.4f},{x['ci_high']:+.4f}] n={x['n_effective_triples']}"

print("=" * 92)
print("FINAL-TOKEN CF2  vs  POOLED (mean-last-5) — the RED-1 replication check")
print("=" * 92)
comp = []
for b in BRANCHES:
    for pop in ("primary", "cross_fitted", "full_A_sensitivity"):
        fb = e.get("CF2_by_stage", {}).get(b, {}).get(pop) or {}
        pb = pooled.get("CF2_by_stage", {}).get(b, {}).get(pop) or {}
        print(f"  {b:15s}{pop:20s} final {fmt(fb):40s} pooled {fmt(pb)}")
        if fb.get("cf2") is not None and pb.get("cf2") is not None:
            comp.append({"stage": b, "population": pop, "final_token_cf2": fb["cf2"],
                         "pooled_cf2": pb["cf2"], "abs_diff": abs(fb["cf2"] - pb["cf2"]),
                         "final_ci": [fb["ci_low"], fb["ci_high"]],
                         "pooled_ci": [pb["ci_low"], pb["ci_high"]]})
    wgb = (e.get("CF2_by_stage", {}).get(b, {}).get("primary") or {}).get("secondary_binary_wildguard") or {}
    if wgb.get("point") is not None:
        print(f"  {b:15s}{'WildGuard(binary)':20s} final {wgb['point']:+.4f} "
              f"[{wgb['ci_low']:+.4f},{wgb['ci_high']:+.4f}] n={wgb['n_effective_triples']}")

if SCOPE == "full":
    xc = e.get("CF2_crossfit_branch_contrasts") or {}
    f2 = xc.get("factorial_2x2") or {}
    print("\ncross-fitted branch contrasts (final-token):")
    for k, p in {**(xc.get("pairwise") or {}), **f2}.items():
        if p.get("estimate") is None: continue
        print(f"  {k:42s} {p['estimate']:+.4f} [{p['ci_low']:+.4f},{p['ci_high']:+.4f}]"
              + ("  <-- CI excl 0" if p.get("ci_excludes_zero") else ""))
    cb = (e.get("CF2_circularity_bias") or {}).get("per_branch") or {}
    print("\ncircularity bias (est_split - cross_fitted, same rows, final-token):")
    for stg, p in cb.items():
        print(f"  {stg:16s} {p['bias_estimation_minus_crossfit']:+.4f} [{p['ci_low']:+.4f},{p['ci_high']:+.4f}]")

# final-token quadrant-C McNemar (regex; reads the held-out files)
print("\nquadrant-C McNemar (final-token, ablated_AD vs ablated_random, soft_deflection):")
for b in BRANCHES:
    hp = f"results/raw/causal_ablation_v2_{b}_L24-28_finaltoken.json"
    if not os.path.exists(hp): continue
    rr = subprocess.run([sys.executable, "-m", "src.analysis.mcnemar_causal_ablation",
                         "--file", hp, "--conditions", f"{b}_ft_ablated_AD", f"{b}_ft_ablated_random",
                         "--quadrant", "C", "--category", "soft_deflection"],
                        capture_output=True, text=True)
    for ln in rr.stdout.splitlines():
        if any(k in ln for k in ("switched", "McNemar", "n=")): print(f"  {b}: {ln.strip()}")

json.dump({"scope": SCOPE, "final_token_endpoints": EP, "rows": comp},
          open(SUM + "/pooled_vs_final_token_CF2.json", "w"), indent=2)
print("\nwrote", SUM + "/pooled_vs_final_token_CF2.json")
print("\n>>> READ: if final-token CF2 primary (M3) has a CI that excludes zero and is")
print(">>> within ~1 CI-width of the pooled +0.114, RED-1 becomes 'robust to the")
print(">>> pooling choice' and the paper flips to final-token primary.")

try:
    subprocess.run(["git", "add", "--",
                    *glob.glob("results/raw/causal_ablation_v2_*finaltoken*"),
                    *glob.glob("results/final_token_repair/summaries/*.json"),
                    *glob.glob("results/final_token_repair/manifests/*.json")], check=True, capture_output=True)
    if subprocess.check_output(["git", "diff", "--cached", "--name-only"], text=True).strip():
        subprocess.run(["git", "commit", "-m", "final-token causal endpoints (Colab)"], check=True, capture_output=True)
        subprocess.run(["git", "pull", "--rebase", "origin", BR], check=False, capture_output=True)
        rr = subprocess.run(["git", "push", "origin", BR], capture_output=True, text=True)
        print("git push rc=", rr.returncode)
except Exception as ex:
    print("git push skipped:", ex, "- Drive has it under", REAL + "/results/")

## DONE — send back
Paste the **cell-8 output**, or download+send
`results/final_token_repair/summaries/{final_token_endpoints,pooled_vs_final_token_CF2}.json`.

In [ ]:
try:
    from google.colab import files
    files.download("results/final_token_repair/summaries/final_token_endpoints.json")
    files.download("results/final_token_repair/summaries/pooled_vs_final_token_CF2.json")
except Exception as ex:
    print("download skipped:", ex, "— Drive UI:", REAL + "/results/final_token_repair/summaries/")